In [ ]:
from pathlib import Path
import json
import pickle
import random
import subprocess
import sys
import warnings
from copy import deepcopy
from dataclasses import dataclass
import mlflow

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
mlflow.set_experiment("Small_Data_Optimization")

In [ ]:
ARTIFACT_DIR = Path("artifacts")
DATASET_PATH = ARTIFACT_DIR / "openml_prepared_datasets.pkl"
SPLIT_PATH = ARTIFACT_DIR / "nested_cv_splits.pkl"
RESULTS_PATH = ARTIFACT_DIR / "evolutionary_algorithm_results.csv"
MASK_ARTIFACT_DIR = ARTIFACT_DIR / "ea_feature_masks"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MASK_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with DATASET_PATH.open("rb") as dataset_file:
    prepared_datasets = pickle.load(dataset_file)

with SPLIT_PATH.open("rb") as split_file:
    nested_cv_splits = pickle.load(split_file)

DATASET_NAMES = list(prepared_datasets.keys())
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
POPULATION_SIZE = 10 #20
GENERATIONS = 20
EVALUATION_BUDGET = POPULATION_SIZE * GENERATIONS
TOURNAMENT_SIZE = 3
N_ELITES = 2
N_SCOUTS = 2
SUCCESS_THRESHOLD = 0.2
FITNESS_PENALTY = 2.0

print(f"Loaded datasets: {DATASET_NAMES}")
print(f"Evaluation budget per fold: {EVALUATION_BUDGET}")

In [ ]:
@dataclass
class Individual:
    n_estimators: int
    learning_rate: float
    max_depth: int
    mask: np.ndarray
    sigma_est: float = 0.2
    sigma_lr: float = 0.35
    sigma_depth: float = 0.2
    mask_flip_rate: float = 0.08
    fitness: float = -np.inf
    mean_acc: float = 0.0
    std_acc: float = 0.0
    parent_fitness: float = -np.inf

    def clone(self) -> "Individual":
        return Individual(
            n_estimators=int(self.n_estimators),
            learning_rate=float(self.learning_rate),
            max_depth=int(self.max_depth),
            mask=self.mask.copy(),
            sigma_est=float(self.sigma_est),
            sigma_lr=float(self.sigma_lr),
            sigma_depth=float(self.sigma_depth),
            mask_flip_rate=float(self.mask_flip_rate),
            fitness=float(self.fitness),
            mean_acc=float(self.mean_acc),
            std_acc=float(self.std_acc),
            parent_fitness=float(self.parent_fitness),
        )

def repair_mask(mask: np.ndarray) -> np.ndarray:
    repaired = mask.astype(bool).copy()
    if not repaired.any():
        repaired[random.randrange(len(repaired))] = True
    return repaired

def random_individual(n_features: int, mutation_state: dict) -> Individual:
    mask = np.random.rand(n_features) < 0.5
    mask = repair_mask(mask)
    return Individual(
        n_estimators=int(np.random.randint(50, 500)),
        learning_rate=float(10 ** np.random.uniform(-3, 0)),
        max_depth=int(np.random.randint(3, 15)),
        mask=mask,
        sigma_est=float(mutation_state["sigma_est"]),
        sigma_lr=float(mutation_state["sigma_lr"]),
        sigma_depth=float(mutation_state["sigma_depth"]),
        mask_flip_rate=float(mutation_state["mask_flip_rate"]),
    )

def log_space_crossover(value_a: float, value_b: float) -> float:
    return float(np.exp((np.log(value_a) + np.log(value_b)) / 2.0))

def crossover(parent_a: Individual, parent_b: Individual) -> Individual:
    child_mask = np.where(np.random.rand(len(parent_a.mask)) < 0.5, parent_a.mask, parent_b.mask).astype(bool)
    child_mask = repair_mask(child_mask)
    child = Individual(
        n_estimators=int((parent_a.n_estimators + parent_b.n_estimators) / 2),
        learning_rate=log_space_crossover(parent_a.learning_rate, parent_b.learning_rate),
        max_depth=int((parent_a.max_depth + parent_b.max_depth) / 2),
        mask=child_mask,
        sigma_est=float((parent_a.sigma_est + parent_b.sigma_est) / 2.0),
        sigma_lr=float((parent_a.sigma_lr + parent_b.sigma_lr) / 2.0),
        sigma_depth=float((parent_a.sigma_depth + parent_b.sigma_depth) / 2.0),
        mask_flip_rate=float((parent_a.mask_flip_rate + parent_b.mask_flip_rate) / 2.0),
    )
    return child

def mutate(individual: Individual, mutation_state: dict, mutation_round: int) -> Individual:
    child = individual.clone()
    
    est_mutation = int(np.random.normal(0, child.n_estimators * child.sigma_est))
    child.n_estimators = int(np.clip(child.n_estimators + est_mutation, 10, 1000))
    
    child.learning_rate = float(np.clip(child.learning_rate * np.exp(np.random.normal(0.0, child.sigma_lr)), 1e-4, 1.0))
    
    depth_mutation = int(np.random.normal(0, child.max_depth * child.sigma_depth))
    child.max_depth = int(np.clip(child.max_depth + depth_mutation, 2, 30))

    flip_mask = np.random.rand(len(child.mask)) < child.mask_flip_rate
    child.mask = np.logical_xor(child.mask, flip_mask)
    child.mask = repair_mask(child.mask)

    if mutation_round > 0:
        jitter = 1.0 + np.random.normal(0.0, 0.02)
        child.sigma_est = float(np.clip(mutation_state["sigma_est"] * jitter, 0.01, 1.0))
        child.sigma_lr = float(np.clip(mutation_state["sigma_lr"] * jitter, 0.01, 1.0))
        child.sigma_depth = float(np.clip(mutation_state["sigma_depth"] * jitter, 0.01, 1.0))
        child.mask_flip_rate = float(np.clip(mutation_state["mask_flip_rate"] * jitter, 1e-3, 0.5))

    return child

def build_model(individual: Individual) -> XGBClassifier:
    return XGBClassifier(
        n_estimators=int(individual.n_estimators),
        learning_rate=float(individual.learning_rate),
        max_depth=int(individual.max_depth),
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

def evaluate_individual(individual: Individual, X: pd.DataFrame, y: np.ndarray, inner_splits: list, feature_names: list) -> tuple[float, float, float]:
    active_mask = individual.mask.astype(bool)
    scores = []

    for split in inner_splits:
        train_idx = np.asarray(split["train_idx"], dtype=int)
        val_idx = np.asarray(split["val_idx"], dtype=int)

        X_train_fold = X.iloc[train_idx, active_mask]
        X_val_fold = X.iloc[val_idx, active_mask]
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_fold)
        X_val_scaled = scaler.transform(X_val_fold)

        model = build_model(individual)
        model.fit(X_train_scaled, y_train_fold)
        predictions = model.predict(X_val_scaled)
        scores.append(accuracy_score(y_val_fold, predictions))

    scores = np.asarray(scores, dtype=float)
    mean_acc = float(scores.mean())
    std_acc = float(scores.std(ddof=0))
    fitness = float(mean_acc - FITNESS_PENALTY * std_acc)
    individual.mean_acc = mean_acc
    individual.std_acc = std_acc
    individual.fitness = fitness
    return mean_acc, std_acc, fitness

def tournament_select(population: list[Individual]) -> Individual:
    contenders = random.sample(population, k=min(TOURNAMENT_SIZE, len(population)))
    return max(contenders, key=lambda ind: ind.fitness)

def update_mutation_state(mutation_state: dict, success_rate: float) -> dict:
    increase_factor = 1.2
    decrease_factor = 0.85
    factor = increase_factor if success_rate > SUCCESS_THRESHOLD else decrease_factor
    return {
        "sigma_est": float(np.clip(mutation_state["sigma_est"] * factor, 0.01, 1.0)),
        "sigma_lr": float(np.clip(mutation_state["sigma_lr"] * factor, 0.01, 1.0)),
        "sigma_depth": float(np.clip(mutation_state["sigma_depth"] * factor, 0.01, 1.0)),
        "mask_flip_rate": float(np.clip(mutation_state["mask_flip_rate"] * factor, 1e-3, 0.5)),
    }

def mask_to_payload(individual: Individual, feature_names: list[str]) -> dict:
    active_mask = individual.mask.astype(bool)
    selected_features = [feature_names[index] for index, is_active in enumerate(active_mask) if is_active]
    return {
        "mask": active_mask.astype(int).tolist(),
        "selected_indices": [int(index) for index, is_active in enumerate(active_mask) if is_active],
        "selected_features": selected_features,
        "selected_feature_count": int(active_mask.sum()),
    }

In [ ]:
def breed_next_population(population: list[Individual], mutation_state: dict, n_features: int) -> tuple[list[Individual], list[Individual]]:
    sorted_population = sorted(population, key=lambda ind: ind.fitness, reverse=True)
    elites = [ind.clone() for ind in sorted_population[:N_ELITES]]

    scouts = [random_individual(n_features, mutation_state) for _ in range(N_SCOUTS)]

    offspring = []
    while len(offspring) < POPULATION_SIZE - N_ELITES - N_SCOUTS:
        parent_a = tournament_select(sorted_population)
        parent_b = tournament_select(sorted_population)
        child = crossover(parent_a, parent_b)
        child.parent_fitness = float(max(parent_a.fitness, parent_b.fitness))
        child = mutate(child, mutation_state, mutation_round=1)
        offspring.append(child)

    next_population = elites + scouts + offspring
    return next_population, offspring
    

def run_evolutionary_search(X: pd.DataFrame, y: np.ndarray, feature_names: list[str], inner_splits: list, random_state: int) -> tuple[Individual, dict, pd.DataFrame]:
    mutation_state = {
        "sigma_est": 0.2,
        "sigma_lr": 0.35,
        "sigma_depth": 0.2,
        "mask_flip_rate": 0.08,
    }

    population = [random_individual(len(feature_names), mutation_state) for _ in range(POPULATION_SIZE)]
    generation_history = []

    for generation in range(1, GENERATIONS + 1):
        for individual in population:
            if not np.isfinite(individual.fitness):
                evaluate_individual(individual, X, y, inner_splits, feature_names)

        population = sorted(population, key=lambda ind: ind.fitness, reverse=True)
        best = population[0]
        mean_fitness = float(np.mean([ind.fitness for ind in population]))

        generation_history.append({
            "generation": generation,
            "best_fitness": best.fitness,
            "best_mean_acc": best.mean_acc,
            "best_std_acc": best.std_acc,
            "population_mean_fitness": mean_fitness,
            "sigma_est": mutation_state["sigma_est"],
            "sigma_lr": mutation_state["sigma_lr"],
            "sigma_depth": mutation_state["sigma_depth"],
            "mask_flip_rate": mutation_state["mask_flip_rate"],
            "best_selected_features": int(best.mask.sum()),
        })

        if generation == GENERATIONS:
            break

        next_population, offspring = breed_next_population(population, mutation_state, len(feature_names))

        for individual in next_population:
            evaluate_individual(individual, X, y, inner_splits, feature_names)

        success_rate = float(np.mean([child.fitness > child.parent_fitness for child in offspring])) if offspring else 0.0
        mutation_state = update_mutation_state(mutation_state, success_rate)

        population = next_population

    final_best = max(population, key=lambda ind: ind.fitness)
    history_df = pd.DataFrame(generation_history)
    return final_best, mutation_state, history_df


def evaluate_on_outer_test(best_individual: Individual, X: pd.DataFrame, y: np.ndarray, outer_train_idx: np.ndarray, outer_test_idx: np.ndarray) -> float:
    active_mask = best_individual.mask.astype(bool)
    X_train = X.iloc[outer_train_idx, active_mask]
    X_test = X.iloc[outer_test_idx, active_mask]
    y_train = y[outer_train_idx]
    y_test = y[outer_test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = build_model(best_individual)
    model.fit(X_train_scaled, y_train)
    return float(accuracy_score(y_test, model.predict(X_test_scaled)))

In [ ]:
results = []

for dataset_name in DATASET_NAMES:
    dataset_payload = prepared_datasets[dataset_name]
    X = dataset_payload["X"]
    y = dataset_payload["y"]
    feature_names = dataset_payload["feature_names"]
    outer_splits = nested_cv_splits[dataset_name]["outer_splits"]

    with mlflow.start_run(run_name=dataset_name):
        mlflow.log_params({
            "dataset": dataset_name,
            "population_size": POPULATION_SIZE,
            "generations": GENERATIONS,
            "evaluation_budget": EVALUATION_BUDGET,
            "outer_folds": N_OUTER_FOLDS,
            "inner_folds": N_INNER_FOLDS,
            "fitness_penalty": FITNESS_PENALTY,
            "selection_operator": "tournament",
            "crossover_operator": "log_space_continuous_uniform_mask",
            "mutation_rule": "Rechenberg_1_5th",
            "scouting": N_SCOUTS,
            "elitism": N_ELITES,
        })

        dataset_fold_results = []

        for outer_fold_info in outer_splits:
            outer_fold = int(outer_fold_info["fold"])
            outer_train_idx = np.asarray(outer_fold_info["train_idx"], dtype=int)
            outer_test_idx = np.asarray(outer_fold_info["test_idx"], dtype=int)
            inner_splits = outer_fold_info["inner_splits"]

            with mlflow.start_run(run_name=f"{dataset_name}_fold_{outer_fold}", nested=True):
                best_individual, mutation_state, history_df = run_evolutionary_search(
                    X=X,
                    y=y,
                    feature_names=feature_names,
                    inner_splits=inner_splits,
                    random_state=RANDOM_STATE + outer_fold,
                )

                test_acc = evaluate_on_outer_test(best_individual, X, y, outer_train_idx, outer_test_idx)
                mask_payload = mask_to_payload(best_individual, feature_names)

                mlflow.log_metrics({
                    "cv_mean": float(best_individual.mean_acc),
                    "cv_std": float(best_individual.std_acc),
                    "test_acc": float(test_acc),
                    "selected_feature_count": float(mask_payload["selected_feature_count"]),
                    "sigma_est": float(best_individual.sigma_est),
                    "sigma_lr": float(best_individual.sigma_lr),
                    "sigma_depth": float(best_individual.sigma_depth),
                    "mask_flip_rate": float(best_individual.mask_flip_rate),
                })
                
                mlflow.log_params({
                    "n_estimators": int(best_individual.n_estimators),
                    "learning_rate": float(best_individual.learning_rate),
                    "max_depth": int(best_individual.max_depth),
                })

                mask_path = MASK_ARTIFACT_DIR / f"{dataset_name}_fold_{outer_fold}_mask.json"
                mask_payload_to_save = {
                    "dataset": dataset_name,
                    "outer_fold": outer_fold,
                    "feature_names": feature_names,
                    "selected_mask": mask_payload["mask"],
                    "selected_indices": mask_payload["selected_indices"],
                    "selected_features": mask_payload["selected_features"],
                    "selected_feature_count": mask_payload["selected_feature_count"],
                }
                mask_path.write_text(json.dumps(mask_payload_to_save, indent=2), encoding="utf-8")
                mlflow.log_artifact(str(mask_path), artifact_path="feature_masks")
                mlflow.log_text(history_df.to_csv(index=False), artifact_file="generation_history.csv")

                fold_result = {
                    "dataset": dataset_name,
                    "outer_fold": outer_fold,
                    "cv_mean": float(best_individual.mean_acc),
                    "cv_std": float(best_individual.std_acc),
                    "test_acc": float(test_acc),
                    "selected_feature_count": int(mask_payload["selected_feature_count"]),
                    "selected_features": json.dumps(mask_payload["selected_features"]),
                    "mask": json.dumps(mask_payload["mask"]),
                    
                    "n_estimators": int(best_individual.n_estimators),
                    "learning_rate": float(best_individual.learning_rate),
                    "max_depth": int(best_individual.max_depth),
                    "sigma_est": float(best_individual.sigma_est),
                    "sigma_lr": float(best_individual.sigma_lr),
                    "sigma_depth": float(best_individual.sigma_depth),
                    "mask_flip_rate": float(best_individual.mask_flip_rate),
                }
                dataset_fold_results.append(fold_result)

        dataset_results_df = pd.DataFrame(dataset_fold_results)
        dataset_results_df.to_csv(RESULTS_PATH.with_name(f"{dataset_name}_evolutionary_results.csv"), index=False)

        mlflow.log_metrics({
            "mean_cv_mean": float(dataset_results_df["cv_mean"].mean()),
            "mean_cv_std": float(dataset_results_df["cv_std"].mean()),
            "mean_test_acc": float(dataset_results_df["test_acc"].mean()),
        })
        mlflow.log_artifact(str(RESULTS_PATH.with_name(f"{dataset_name}_evolutionary_results.csv")))

        results.extend(dataset_fold_results)

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_PATH, index=False)
results_df